[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/083.%20The%20Multi-Facility%20Location%20-%20p-Median%20Problem/P83-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 83. The Multi-Facility Location: p-Median Problem

## Tier 4: The AI/ML/RL Augmentation Method (Ensemble Learning Framework)

### Key assumptions
- Historical problem instances contain learnable patterns
- Multiple ML models can capture different aspects of facility selection
- Ensemble methods improve prediction accuracy through diversity
- ML-guided initialization enhances heuristic performance

### Approach (step-by-step)
The ensemble learning framework integrates multiple predictive models:
1. **Data Generation**: Create training instances with known optimal solutions
2. **Feature Engineering**: Extract meaningful problem characteristics
3. **Model Training**: Train Random Forest, Gradient Boosting, and Neural Network
4. **Prediction**: Use ensemble to predict facility selection probabilities
5. **ML-Guided Search**: Initialize and guide heuristic search with ML predictions
6. **Performance Evaluation**: Compare ML-guided vs random initialization

### What to look for in the results
- Model training performance and accuracy metrics
- Feature importance analysis from ensemble models
- ML-guided initialization quality improvement
- Ensemble vs individual model performance comparison
- Real-world applicability and scalability analysis

### Concrete example (from the source)
Clustered 20-customer, 30-facility problem demonstrating ensemble learning with 200 training samples, achieving 24.69% improvement over random initialization.

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict, Set
import random
import time
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for p-Median Ensemble Learning")

In [ ]:
class PMedianEnsembleLearning:
    """
    Ensemble Learning framework for p-Median problem
    Integrates multiple ML models to guide heuristic search
    """
    
    def __init__(self, customer_positions: List[float], demands: List[float], 
                 facility_positions: List[float], p: int):
        """
        Initialize p-Median Ensemble Learning solver
        
        Args:
            customer_positions: Geographic positions of customers
            demands: Demand volumes for each customer
            facility_positions: Potential facility locations
            p: Number of facilities to locate
        """
        self.customer_positions = np.array(customer_positions)
        self.demands = np.array(demands)
        self.facility_positions = np.array(facility_positions)
        self.p = p
        self.n_customers = len(customer_positions)
        self.n_facilities = len(facility_positions)
        
        # Calculate distance matrix
        self.distances = self._calculate_distances()
        
        # Ensemble models
        self.rf_classifier = None  # Random Forest for facility selection
        self.gb_regressor = None   # Gradient Boosting for cost prediction
        self.nn_regressor = None   # Neural Network for cost prediction
        
        # Training data
        self.training_data = None
        self.feature_names = None
        
        # Solution tracking
        self.best_solution = None
        self.best_cost = np.inf
        
    def _calculate_distances(self) -> np.ndarray:
        """Calculate Euclidean distance matrix between customers and facilities"""
        distances = np.zeros((self.n_customers, self.n_facilities))
        for i in range(self.n_customers):
            for j in range(self.n_facilities):
                distances[i, j] = abs(self.customer_positions[i] - self.facility_positions[j])
        return distances
    
    def _assign_customers(self, facility_set: Set[int]) -> Dict[int, List[int]]:
        """Assign each customer to nearest open facility"""
        assignments = {fac_idx: [] for fac_idx in facility_set}
        
        for cust_idx in range(self.n_customers):
            min_dist = np.inf
            nearest_facility = -1
            
            for fac_idx in facility_set:
                dist = self.distances[cust_idx, fac_idx]
                if dist < min_dist:
                    min_dist = dist
                    nearest_facility = fac_idx
            
            assignments[nearest_facility].append(cust_idx)
        
        return assignments
    
    def _calculate_total_cost(self, facility_set: Set[int]) -> float:
        """Calculate total transportation cost for given facility set"""
        assignments = self._assign_customers(facility_set)
        total_cost = 0.0
        
        for fac_idx, customers in assignments.items():
            for cust_idx in customers:
                total_cost += self.demands[cust_idx] * self.distances[cust_idx, fac_idx]
        
        return total_cost
    
    def _extract_features(self, customer_positions: np.ndarray, demands: np.ndarray, 
                         facility_positions: np.ndarray, facility_idx: int) -> np.ndarray:
        """
        Extract features for a specific facility
        
        Args:
            customer_positions: Customer positions
            demands: Customer demands
            facility_positions: Facility positions
            facility_idx: Index of facility to extract features for
            
        Returns:
            Feature vector for the facility
        """
        features = []
        fac_pos = facility_positions[facility_idx]
        
        # Geographic features
        features.append(fac_pos)  # Facility position
        features.append(np.mean(customer_positions))  # Mean customer position
        features.append(np.std(customer_positions))  # Customer position dispersion
        features.append(fac_pos - np.mean(customer_positions))  # Relative position
        
        # Demand-based features
        total_demand = np.sum(demands)
        features.append(total_demand)  # Total demand
        features.append(np.mean(demands))  # Mean demand
        features.append(np.std(demands))  # Demand variance
        
        # Distance-based features
        distances_to_facility = np.abs(customer_positions - fac_pos)
        features.append(np.mean(distances_to_facility))  # Mean distance to customers
        features.append(np.min(distances_to_facility))  # Min distance to customers
        features.append(np.max(distances_to_facility))  # Max distance to customers
        
        # Coverage features
        nearby_customers = np.sum(distances_to_facility < np.mean(distances_to_facility))
        features.append(nearby_customers)  # Number of nearby customers
        features.append(nearby_customers / len(customer_positions))  # Coverage ratio
        
        # Centrality features
        all_distances = []
        for other_fac in range(len(facility_positions)):
            if other_fac != facility_idx:
                all_distances.append(abs(fac_pos - facility_positions[other_fac]))
        
        if all_distances:
            features.append(np.mean(all_distances))  # Mean distance to other facilities
            features.append(np.min(all_distances))  # Min distance to other facilities
        else:
            features.append(0)  # Mean distance to other facilities
            features.append(0)  # Min distance to other facilities
        
        # Demand-weighted distance features
        weighted_distances = demands * distances_to_facility
        features.append(np.sum(weighted_distances))  # Total weighted distance
        features.append(np.mean(weighted_distances))  # Mean weighted distance
        
        return np.array(features)
    
    def _generate_training_instance(self, n_customers: int, n_facilities: int, p: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Set[int], float]:
        """
        Generate a single training instance with known optimal solution
        
        Returns:
            Tuple of (customer_positions, demands, facility_positions, optimal_facilities, optimal_cost)
        """
        # Generate random problem instance
        cust_positions = np.random.uniform(0, 50, n_customers)
        cust_positions.sort()
        dem = np.random.uniform(5, 25, n_customers)
        fac_positions = np.random.uniform(0, 50, n_facilities)
        fac_positions.sort()
        
        # Find near-optimal solution using simple heuristic (for training)
        best_cost = np.inf
        best_facilities = set()
        
        # Try multiple random initializations and pick the best
        for _ in range(20):
            candidate_facilities = set(random.sample(range(n_facilities), p))
            
            # Calculate cost
            assignments = {fac_idx: [] for fac_idx in candidate_facilities}
            for cust_idx in range(n_customers):
                min_dist = np.inf
                nearest_facility = -1
                for fac_idx in candidate_facilities:
                    dist = abs(cust_positions[cust_idx] - fac_positions[fac_idx])
                    if dist < min_dist:
                        min_dist = dist
                        nearest_facility = fac_idx
                assignments[nearest_facility].append(cust_idx)
            
            cost = 0.0
            for fac_idx, customers in assignments.items():
                for cust_idx in customers:
                    cost += dem[cust_idx] * abs(cust_positions[cust_idx] - fac_positions[fac_idx])
            
            if cost < best_cost:
                best_cost = cost
                best_facilities = candidate_facilities.copy()
        
        return cust_positions, dem, fac_positions, best_facilities, best_cost
    
    def _generate_training_data(self, n_instances: int = 200) -> None:
        """
        Generate training data for ensemble models
        
        Args:
            n_instances: Number of training instances to generate
        """
        print(f"Generating {n_instances} training instances...")
        
        all_features = []
        all_labels = []  # For classification (facility selection)
        all_costs = []   # For regression (cost prediction)
        
        # Vary problem sizes for better generalization
        for i in range(n_instances):
            if i % 50 == 0 and i > 0:
                print(f"Generated {i}/{n_instances} samples")
            
            # Generate problem instance with varying size
            n_cust = random.randint(8, 15)
            n_fac = random.randint(12, 20)
            p_val = random.randint(2, min(5, n_fac // 3))
            
            cust_positions, dem, fac_positions, optimal_facilities, optimal_cost = self._generate_training_instance(
                n_cust, n_fac, p_val
            )
            
            # Extract features for each facility
            for fac_idx in range(n_fac):
                features = self._extract_features(cust_positions, dem, fac_positions, fac_idx)
                all_features.append(features)
                
                # Classification label: is this facility in optimal solution?
                label = 1 if fac_idx in optimal_facilities else 0
                all_labels.append(label)
                
                # Regression target: contribution to total cost if selected
                if fac_idx in optimal_facilities:
                    # Calculate this facility's cost contribution
                    assignments = self._assign_customers(optimal_facilities)
                    facility_customers = assignments[fac_idx]
                    cost_contribution = sum(dem[c] * abs(cust_positions[c] - fac_positions[fac_idx]) for c in facility_customers)
                    all_costs.append(cost_contribution)
                else:
                    all_costs.append(0)  # Not selected, no contribution
        
        print(f"Generated {n_instances}/{n_instances} samples")
        
        # Store training data
        self.training_data = {
            'features': np.array(all_features),
            'classification_labels': np.array(all_labels),
            'regression_targets': np.array(all_costs)
        }
        
        # Define feature names for interpretation
        self.feature_names = [
            'facility_position', 'mean_customer_pos', 'customer_pos_std', 'relative_position',
            'total_demand', 'mean_demand', 'demand_std',
            'mean_distance', 'min_distance', 'max_distance',
            'nearby_customers', 'coverage_ratio',
            'mean_facility_distance', 'min_facility_distance',
            'total_weighted_distance', 'mean_weighted_distance'
        ]
    
    def train_ensemble_models(self) -> None:
        """Train the ensemble learning models"""
        if self.training_data is None:
            raise ValueError("Training data not generated. Call _generate_training_data() first.")
        
        print("Training ensemble models...")
        
        X = self.training_data['features']
        y_class = self.training_data['classification_labels']
        y_reg = self.training_data['regression_targets']
        
        # Split data for validation
        X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
            X, y_class, y_reg, test_size=0.2, random_state=42
        )
        
        # Train Random Forest Classifier
        print("Training Random Forest Classifier...")
        self.rf_classifier = RandomForestClassifier(
            n_estimators=100, max_depth=10, random_state=42
        )
        self.rf_classifier.fit(X_train, y_class_train)
        
        rf_accuracy = accuracy_score(y_class_test, self.rf_classifier.predict(X_test))
        print(f"Random Forest accuracy: {rf_accuracy:.3f}")
        
        # Train Gradient Boosting Regressor
        print("Training Gradient Boosting Regressor...")
        self.gb_regressor = GradientBoostingRegressor(
            n_estimators=100, max_depth=6, random_state=42
        )
        self.gb_regressor.fit(X_train, y_reg_train)
        
        gb_mse = mean_squared_error(y_reg_test, self.gb_regressor.predict(X_test))
        print(f"Gradient Boosting MSE: {gb_mse:.3f}")
        
        # Train Neural Network Regressor
        print("Training Neural Network Regressor...")
        self.nn_regressor = MLPRegressor(
            hidden_layer_sizes=(64, 32), max_iter=500, random_state=42
        )
        self.nn_regressor.fit(X_train, y_reg_train)
        
        nn_mse = mean_squared_error(y_reg_test, self.nn_regressor.predict(X_test))
        print(f"Neural Network MSE: {nn_mse:.3f}")
    
    def predict_facility_probabilities(self) -> np.ndarray:
        """
        Predict facility selection probabilities using trained ensemble
        
        Returns:
            Array of facility selection probabilities
        """
        if self.rf_classifier is None:
            raise ValueError("Models not trained. Call train_ensemble_models() first.")
        
        # Extract features for current problem
        features = []
        for fac_idx in range(self.n_facilities):
            feat = self._extract_features(
                self.customer_positions, self.demands, self.facility_positions, fac_idx
            )
            features.append(feat)
        
        features = np.array(features)
        
        # Get probabilities from Random Forest
        probabilities = self.rf_classifier.predict_proba(features)[:, 1]  # Probability of being selected
        
        return probabilities
    
    def predict_solution_costs(self, facility_sets: List[Set[int]]) -> np.ndarray:
        """
        Predict costs for multiple facility sets using ensemble
        
        Args:
            facility_sets: List of facility sets to evaluate
            
        Returns:
            Array of predicted costs
        """
        if self.gb_regressor is None or self.nn_regressor is None:
            raise ValueError("Models not trained. Call train_ensemble_models() first.")
        
        predicted_costs = []
        
        for facility_set in facility_sets:
            # Extract features for selected facilities
            total_cost_pred = 0
            
            for fac_idx in facility_set:
                feat = self._extract_features(
                    self.customer_positions, self.demands, self.facility_positions, fac_idx
                )
                feat = feat.reshape(1, -1)
                
                # Ensemble prediction (average of GB and NN)
                gb_pred = self.gb_regressor.predict(feat)[0]
                nn_pred = self.nn_regressor.predict(feat)[0]
                ensemble_pred = (gb_pred + nn_pred) / 2
                
                total_cost_pred += ensemble_pred
            
            predicted_costs.append(total_cost_pred)
        
        return np.array(predicted_costs)
    
    def generate_ml_guided_initial_solution(self) -> Set[int]:
        """
        Generate initial solution guided by ML predictions
        
        Returns:
            ML-guided initial facility set
        """
        probabilities = self.predict_facility_probabilities()
        
        # Select top-p facilities based on probabilities
        top_indices = np.argsort(probabilities)[-self.p:]
        
        return set(top_indices)
    
    def solve_with_ml_guidance(self, max_iterations: int = 50, verbose: bool = True) -> Tuple[float, Set[int], Dict[int, List[int]]]:
        """
        Solve p-Median problem using ML-guided local search
        
        Args:
            max_iterations: Maximum iterations for local search
            verbose: Whether to print progress
            
        Returns:
            Tuple of (total_cost, facility_locations, customer_assignments)
        """
        if verbose:
            print(f"Solving p-Median with ML-Guided Local Search")
            print(f"Customers: {self.n_customers}, Facilities: {self.n_facilities}, p: {self.p}")
        
        # Generate ML-guided initial solution
        current_solution = self.generate_ml_guided_initial_solution()
        current_cost = self._calculate_total_cost(current_solution)
        
        if verbose:
            print(f"\nML-guided initial solution: {sorted(current_solution)}")
            print(f"ML-guided initial cost: {current_cost:.2f}")
        
        # Improve with local search
        best_solution = current_solution.copy()
        best_cost = current_cost
        
        for iteration in range(max_iterations):
            best_improvement = 0
            best_swap = (-1, -1)
            
            # Try all possible swaps
            for facility_out in current_solution:
                for facility_in in range(self.n_facilities):
                    if facility_in in current_solution:
                        continue
                    
                    new_facilities = current_solution.copy()
                    new_facilities.remove(facility_out)
                    new_facilities.add(facility_in)
                    new_cost = self._calculate_total_cost(new_facilities)
                    improvement = current_cost - new_cost
                    
                    if improvement > best_improvement:
                        best_improvement = improvement
                        best_swap = (facility_out, facility_in)
            
            if best_improvement > 0.001:
                current_solution.remove(best_swap[0])
                current_solution.add(best_swap[1])
                current_cost -= best_improvement
                
                if current_cost < best_cost:
                    best_solution = current_solution.copy()
                    best_cost = current_cost
            else:
                break
        
        # Get final assignments
        final_assignments = self._assign_customers(best_solution)
        
        if verbose:
            print(f"\nImproved solution: {sorted(best_solution)}")
            print(f"Improved cost: {best_cost:.2f}")
        
        self.best_solution = best_solution
        self.best_cost = best_cost
        
        return best_cost, best_solution, final_assignments
    
    def compare_with_random_baseline(self, n_trials: int = 10) -> Dict:
        """
        Compare ML-guided approach with random initialization
        
        Args:
            n_trials: Number of random trials for comparison
            
        Returns:
            Comparison results
        """
        print(f"\nComparing with random baseline ({n_trials} trials)...")
        
        # ML-guided solution
        ml_cost, ml_facilities, ml_assignments = self.solve_with_ml_guidance(verbose=False)
        
        # Random baseline solutions
        random_costs = []
        
        for trial in range(n_trials):
            random_facilities = set(random.sample(range(self.n_facilities), self.p))
            random_cost = self._calculate_total_cost(random_facilities)
            random_costs.append(random_cost)
        
        # Calculate statistics
        mean_random_cost = np.mean(random_costs)
        std_random_cost = np.std(random_costs)
        min_random_cost = np.min(random_costs)
        
        improvement_over_mean = ((mean_random_cost - ml_cost) / mean_random_cost) * 100
        improvement_over_best = ((min_random_cost - ml_cost) / min_random_cost) * 100
        
        results = {
            'ml_cost': ml_cost,
            'ml_facilities': sorted(ml_facilities),
            'mean_random_cost': mean_random_cost,
            'std_random_cost': std_random_cost,
            'min_random_cost': min_random_cost,
            'improvement_over_mean': improvement_over_mean,
            'improvement_over_best': improvement_over_best,
            'random_costs': random_costs
        }
        
        print(f"ML-guided cost: {ml_cost:.2f}")
        print(f"Random baseline cost: {mean_random_cost:.2f} ± {std_random_cost:.2f}")
        print(f"Best random cost: {min_random_cost:.2f}")
        print(f"ML improvement over random: {improvement_over_mean:.2f}%")
        
        return results
    
    def print_solution(self, total_cost: float, facility_locations: Set[int], 
                      customer_assignments: Dict[int, List[int]]):
        """Print the solution in a readable format"""
        print(f"\n{'='*60}")
        print("ENSEMBLE LEARNING SOLUTION")
        print(f"{'='*60}")
        print(f"Total Transportation Cost: {total_cost:.2f}")
        print(f"Number of Facilities: {self.p}")
        print(f"\nFacility Locations and Assignments:")
        
        for i, fac_idx in enumerate(sorted(facility_locations)):
            fac_pos = self.facility_positions[fac_idx]
            customers = customer_assignments[fac_idx]
            customer_positions = [self.customer_positions[c] for c in customers]
            customer_demands = [self.demands[c] for c in customers]
            
            print(f"\n  Facility {i+1} at position {fac_pos:.1f} (Index {fac_idx}):")
            print(f"    Serves customers at positions: {customer_positions}")
            print(f"    Customer demands: {customer_demands}")
            
            # Calculate cost for this facility
            cost = sum(self.demands[c] * self.distances[c, fac_idx] for c in customers)
            print(f"    Cost contribution: {cost:.2f}")
            print(f"    Number of customers: {len(customers)}")

print("PMedianEnsembleLearning class defined successfully")

PMedianEnsembleLearning class defined successfully


In [ ]:
try:
    # Create the concrete example from the source
    # Clustered 20-customer, 30-facility problem
    
    # Generate clustered test data
    n_customers = 20
    n_facilities = 30
    p = 5
    
    # Create clustered customer distribution
    np.random.seed(42)
    clusters = 3
    customers_per_cluster = n_customers // clusters
    
    customer_positions = []
    demands = []
    
    for cluster_id in range(clusters):
        # Cluster center
        cluster_center = cluster_id * 15 + np.random.uniform(-2, 2)
        
        # Generate customers around cluster center
        for i in range(customers_per_cluster):
            pos = cluster_center + np.random.normal(0, 3)
            customer_positions.append(max(0, min(50, pos)))  # Keep within bounds
            demands.append(np.random.uniform(8, 30))
    
    # Add remaining customers
    remaining = n_customers - len(customer_positions)
    for i in range(remaining):
        customer_positions.append(np.random.uniform(0, 50))
        demands.append(np.random.uniform(8, 30))
    
    customer_positions = np.array(customer_positions)
    customer_positions.sort()
    demands = np.array(demands)
    
    # Generate facility positions (more spread out)
    facility_positions = np.random.uniform(0, 50, n_facilities)
    facility_positions.sort()
    
    print("CONCRETE EXAMPLE SETUP")
    print(f"Number of customers: {n_customers}")
    print(f"Number of potential facilities: {n_facilities}")
    print(f"Number of facilities to locate: {p}")
    print(f"\nCustomer positions: {customer_positions.round(2).tolist()}")
    print(f"Customer demands: {demands.round(2).tolist()}")
    print(f"Facility positions: {facility_positions.round(2).tolist()}")
    
    # Create and solve the problem
    solver = PMedianEnsembleLearning(customer_positions.tolist(), demands.tolist(), 
                                    facility_positions.tolist(), p)
    
    # Generate training data and train models
    solver._generate_training_data(n_instances=200)
    solver.train_ensemble_models()
    
    # Solve with ML guidance
    total_cost, facility_locations, customer_assignments = solver.solve_with_ml_guidance(
        max_iterations=50, verbose=True
    )
    
    # Print detailed solution
    solver.print_solution(total_cost, facility_locations, customer_assignments)
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    # Compare with random baseline
    comparison_results = solver.compare_with_random_baseline(n_trials=10)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'solver' is not defined...]

In [ ]:
try:
    # Visualize ensemble learning results
    def visualize_ensemble_learning(solver, comparison_results):
        """Visualize the ensemble learning results and analysis"""
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        # Plot 1: Model training performance
        if solver.training_data:
            # Extract feature importance from Random Forest
            feature_importance = solver.rf_classifier.feature_importances_
            feature_names = solver.feature_names
            
            # Sort by importance
            sorted_idx = np.argsort(feature_importance)[-10:]
            sorted_importance = feature_importance[sorted_idx]
            sorted_names = [feature_names[i] for i in sorted_idx]
            
            bars = ax1.barh(range(len(sorted_names)), sorted_importance, alpha=0.8)
            ax1.set_yticks(range(len(sorted_names)))
            ax1.set_yticklabels(sorted_names)
            ax1.set_xlabel('Feature Importance', fontsize=12)
            ax1.set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
            ax1.grid(True, alpha=0.3, axis='x')
        
        # Plot 2: ML vs Random baseline comparison
        methods = ['ML-Guided', 'Random Mean', 'Random Best']
        costs = [comparison_results['ml_cost'], 
                 comparison_results['mean_random_cost'], 
                 comparison_results['min_random_cost']]
        colors = ['green', 'blue', 'orange']
        
        bars = ax2.bar(methods, costs, alpha=0.8, color=colors)
        ax2.set_ylabel('Total Cost', fontsize=12)
        ax2.set_title('ML-Guided vs Random Baseline', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        # Add value labels
        for bar, cost in zip(bars, costs):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + max(costs)*0.01,
                    f'{cost:.1f}', ha='center', va='bottom', fontweight='bold')
        
        # Add improvement annotation
        improvement = comparison_results['improvement_over_mean']
        ax2.text(0.5, 0.95, f'ML Improvement: {improvement:.1f}%', transform=ax2.transAxes,
                ha='center', va='top', fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        
        # Plot 3: Random baseline distribution
        random_costs = comparison_results['random_costs']
        ml_cost = comparison_results['ml_cost']
        
        ax3.hist(random_costs, bins=15, alpha=0.7, color='blue', edgecolor='black')
        ax3.axvline(ml_cost, color='red', linewidth=3, linestyle='--', label='ML-Guided')
        ax3.axvline(np.mean(random_costs), color='orange', linewidth=2, linestyle='-', label='Random Mean')
        ax3.set_xlabel('Total Cost', fontsize=12)
        ax3.set_ylabel('Frequency', fontsize=12)
        ax3.set_title('Random Baseline Cost Distribution', fontsize=14, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Facility selection probabilities
        probabilities = solver.predict_facility_probabilities()
        selected_facilities = list(solver.best_solution)
        
        facility_indices = range(len(probabilities))
        colors = ['red' if i in selected_facilities else 'lightblue' for i in facility_indices]
        
        bars = ax4.bar(facility_indices, probabilities, alpha=0.7, color=colors, edgecolor='black')
        ax4.set_xlabel('Facility Index', fontsize=12)
        ax4.set_ylabel('Selection Probability', fontsize=12)
        ax4.set_title('ML Facility Selection Probabilities', fontsize=14, fontweight='bold')
        ax4.grid(True, alpha=0.3, axis='y')
        
        # Add legend
        import matplotlib.patches as mpatches
        red_patch = mpatches.Patch(color='red', alpha=0.7, label='Selected')
        blue_patch = mpatches.Patch(color='lightblue', alpha=0.7, label='Not Selected')
        ax4.legend(handles=[red_patch, blue_patch])
        
        plt.tight_layout()
        plt.show()
    
    # Visualize results
    visualize_ensemble_learning(solver, comparison_results)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'solver' is not defined...]

In [ ]:
try:
    # Visualize solution layout
    def visualize_ml_solution_layout(customer_positions, demands, facility_positions, 
                                   facility_locations, customer_assignments, total_cost):
        """Visualize the ML-guided solution layout"""
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        
        # Plot 1: Geographic layout with assignments
        ax1.set_title('ML-Guided Solution: Facility Locations and Customer Assignments', 
                     fontsize=14, fontweight='bold')
        
        # Plot customers with size proportional to demand
        for i, (pos, demand) in enumerate(zip(customer_positions, demands)):
            ax1.scatter(pos, 0, s=demand*6, c='blue', alpha=0.7, edgecolors='black', linewidth=1)
            ax1.annotate(f'C{i+1}', (pos, 0), xytext=(pos, 0.6), 
                        ha='center', va='bottom', fontsize=8)
        
        # Plot all potential facilities
        for pos in facility_positions:
            ax1.scatter(pos, 0, s=50, c='lightgray', marker='^', alpha=0.5)
        
        # Plot selected facilities and assignments
        colors = ['red', 'green', 'orange', 'purple', 'brown']
        
        for i, fac_idx in enumerate(sorted(facility_locations)):
            fac_pos = facility_positions[fac_idx]
            color = colors[i % len(colors)]
            customers = customer_assignments[fac_idx]
            
            # Plot facility
            ax1.scatter(fac_pos, 0, s=150, c=color, marker='^', edgecolors='black', linewidth=2)
            ax1.annotate(f'F{fac_idx}', (fac_pos, 0), xytext=(fac_pos, -0.6), 
                        ha='center', va='top', fontsize=9, fontweight='bold')
            
            # Draw assignment lines
            for cust_idx in customers:
                cust_pos = customer_positions[cust_idx]
                ax1.plot([fac_pos, cust_pos], [0, 0], color=color, alpha=0.6, linewidth=1.5)
        
        ax1.set_xlabel('Geographic Position', fontsize=12)
        ax1.set_ylabel('Location', fontsize=12)
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(-1.2, 1.2)
        
        # Plot 2: Performance analysis
        ax2.set_title('ML Ensemble Performance Analysis', fontsize=14, fontweight='bold')
        
        # Create performance metrics
        facility_data = []
        facility_labels = []
        
        for i, fac_idx in enumerate(sorted(facility_locations)):
            customers = customer_assignments[fac_idx]
            
            # Calculate metrics
            cost = sum(demands[c] * abs(customer_positions[c] - facility_positions[fac_idx]) for c in customers)
            total_demand = sum(demands[c] for c in customers)
            avg_distance = sum(abs(customer_positions[c] - facility_positions[fac_idx]) for c in customers) / len(customers) if customers else 0
            
            facility_data.append({
                'cost': cost,
                'demand': total_demand,
                'customers': len(customers),
                'avg_distance': avg_distance
            })
            facility_labels.append(f'F{fac_idx}\n({len(customers)} cust)')
        
        # Create grouped visualization
        x = np.arange(len(facility_labels))
        width = 0.2
        
        # Normalize values for better comparison
        max_cost = max(d['cost'] for d in facility_data)
        max_demand = max(d['demand'] for d in facility_data)
        max_customers = max(d['customers'] for d in facility_data)
        max_distance = max(d['avg_distance'] for d in facility_data)
        
        costs_norm = [d['cost']/max_cost for d in facility_data]
        demands_norm = [d['demand']/max_demand for d in facility_data]
        customers_norm = [d['customers']/max_customers for d in facility_data]
        distances_norm = [d['avg_distance']/max_distance for d in facility_data]
        
        bars1 = ax2.bar(x - width*1.5, costs_norm, width, label='Cost (norm)', alpha=0.8)
        bars2 = ax2.bar(x - width*0.5, demands_norm, width, label='Demand (norm)', alpha=0.8)
        bars3 = ax2.bar(x + width*0.5, customers_norm, width, label='Customers (norm)', alpha=0.8)
        bars4 = ax2.bar(x + width*1.5, distances_norm, width, label='Avg Distance (norm)', alpha=0.8)
        
        ax2.set_xlabel('Facilities', fontsize=12)
        ax2.set_ylabel('Normalized Value', fontsize=12)
        ax2.set_xticks(x)
        ax2.set_xticklabels(facility_labels)
        ax2.legend()
        ax2.grid(True, alpha=0.3, axis='y')
        
        # Add total cost annotation
        ax2.text(0.02, 0.98, f'Total Cost: {total_cost:.2f}\nML Improvement: {comparison_results["improvement_over_mean"]:.1f}%', 
                transform=ax2.transAxes, ha='left', va='top', fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        
        plt.tight_layout()
        plt.show()
    
    # Create solution visualization
    visualize_ml_solution_layout(customer_positions, demands, facility_positions, 
                              facility_locations, customer_assignments, total_cost)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'facility_positions' is not defined...]

In [ ]:
try:
    # Analyze ensemble model performance
    def analyze_ensemble_performance():
        """Analyze the performance of individual ensemble models"""
        print("\n" + "="*60)
        print("ANALYZING ENSEMBLE MODEL PERFORMANCE")
        print("="*60)
        
        if not solver.training_data:
            print("No training data available for analysis")
            return
        
        # Test on a held-out set
        X = solver.training_data['features']
        y_class = solver.training_data['classification_labels']
        y_reg = solver.training_data['regression_targets']
        
        # Split for testing
        _, X_test, _, y_class_test, _, y_reg_test = train_test_split(
            X, y_class, y_reg, test_size=0.2, random_state=42
        )
        
        # Get predictions from each model
        rf_pred = solver.rf_classifier.predict(X_test)
        rf_proba = solver.rf_classifier.predict_proba(X_test)[:, 1]
        gb_pred = solver.gb_regressor.predict(X_test)
        nn_pred = solver.nn_regressor.predict(X_test)
        
        # Ensemble regression prediction
        ensemble_pred = (gb_pred + nn_pred) / 2
        
        # Calculate metrics
        rf_accuracy = accuracy_score(y_class_test, rf_pred)
        gb_mse = mean_squared_error(y_reg_test, gb_pred)
        nn_mse = mean_squared_error(y_reg_test, nn_pred)
        ensemble_mse = mean_squared_error(y_reg_test, ensemble_pred)
        
        print(f"\nModel Performance Metrics:")
        print(f"Random Forest Classification Accuracy: {rf_accuracy:.3f}")
        print(f"Gradient Boosting MSE: {gb_mse:.3f}")
        print(f"Neural Network MSE: {nn_mse:.3f}")
        print(f"Ensemble Regression MSE: {ensemble_mse:.3f}")
        print(f"Ensemble Improvement over GB: {((gb_mse - ensemble_mse) / gb_mse * 100):.1f}%")
        print(f"Ensemble Improvement over NN: {((nn_mse - ensemble_mse) / nn_mse * 100):.1f}%")
        
        # Create performance comparison visualization
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Classification accuracy
        ax1.bar(['Random Forest'], [rf_accuracy], alpha=0.8, color='blue')
        ax1.set_ylabel('Accuracy', fontsize=12)
        ax1.set_title('Classification Performance', fontsize=14, fontweight='bold')
        ax1.set_ylim([0, 1])
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Add value label
        ax1.text(0, rf_accuracy + 0.02, f'{rf_accuracy:.3f}', ha='center', va='bottom', fontweight='bold')
        
        # Plot 2: Regression MSE comparison
        models = ['Gradient Boosting', 'Neural Network', 'Ensemble']
        mse_values = [gb_mse, nn_mse, ensemble_mse]
        colors = ['green', 'orange', 'red']
        
        bars = ax2.bar(models, mse_values, alpha=0.8, color=colors)
        ax2.set_ylabel('Mean Squared Error', fontsize=12)
        ax2.set_title('Regression Performance', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        # Add value labels
        for bar, mse in zip(bars, mse_values):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + max(mse_values)*0.01,
                    f'{mse:.3f}', ha='center', va='bottom', fontweight='bold')
        
        # Plot 3: Prediction distribution (regression)
        ax3.hist(y_reg_test, bins=20, alpha=0.5, color='blue', label='True', density=True)
        ax3.hist(ensemble_pred, bins=20, alpha=0.5, color='red', label='Ensemble Pred', density=True)
        ax3.set_xlabel('Cost Contribution', fontsize=12)
        ax3.set_ylabel('Density', fontsize=12)
        ax3.set_title('Prediction Distribution', fontsize=14, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Feature importance deep dive
        feature_importance = solver.rf_classifier.feature_importances_
        feature_names = solver.feature_names
        
        # Group features by category
        geographic_features = ['facility_position', 'mean_customer_pos', 'customer_pos_std', 'relative_position']
        demand_features = ['total_demand', 'mean_demand', 'demand_std']
        distance_features = ['mean_distance', 'min_distance', 'max_distance']
        coverage_features = ['nearby_customers', 'coverage_ratio']
        centrality_features = ['mean_facility_distance', 'min_facility_distance']
        weighted_features = ['total_weighted_distance', 'mean_weighted_distance']
        
        categories = {
            'Geographic': geographic_features,
            'Demand': demand_features,
            'Distance': distance_features,
            'Coverage': coverage_features,
            'Centrality': centrality_features,
            'Weighted': weighted_features
        }
        
        category_importance = {}
        for cat_name, feat_list in categories.items():
            cat_importance = 0
            for feat_name in feat_list:
                if feat_name in feature_names:
                    idx = feature_names.index(feat_name)
                    cat_importance += feature_importance[idx]
            category_importance[cat_name] = cat_importance
        
        # Plot category importance
        cat_names = list(category_importance.keys())
        cat_values = list(category_importance.values())
        
        bars = ax4.bar(cat_names, cat_values, alpha=0.8, color='purple')
        ax4.set_ylabel('Cumulative Importance', fontsize=12)
        ax4.set_title('Feature Category Importance', fontsize=14, fontweight='bold')
        ax4.grid(True, alpha=0.3, axis='y')
        plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
        
        # Add value labels
        for bar, value in zip(bars, cat_values):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + max(cat_values)*0.01,
                    f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    # Analyze ensemble performance
    analyze_ensemble_performance()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'solver' is not defined...]

### Why this Tier exists vs earlier Tiers
This Tier provides **data-driven intelligence** for facility location:
- **Learning from experience** using historical problem instances
- **Pattern recognition** for optimal facility characteristics
- **Ensemble wisdom** combining multiple ML approaches
- **Intelligent initialization** improving heuristic performance
- **Adaptive solutions** that learn from problem structure

### Pros / Cons vs earlier Tiers
**Advantages over Tier 1 (Dynamic Programming):**
- Scales to much larger problem instances (100+ customers)
- Learns from data rather than exhaustive computation
- Adapts to specific problem patterns and characteristics
- Provides probabilistic guidance for decision making

**Advantages over Tier 2 (Local Search):**
- Intelligent initialization reduces search time
- Higher quality solutions through learned patterns
- Feature importance insights for problem understanding
- Ensemble robustness reduces prediction errors

**Advantages over Tier 3 (Tabu Search):**
- Data-driven guidance vs memory-based search
- Predictive capabilities for solution quality estimation
- Transfer learning potential across problem instances
- Explainable insights through feature analysis

**Disadvantages:**
- Requires training data generation and model training time
- Performance depends on training data quality and diversity
- More complex implementation with ML dependencies
- May overfit to specific problem patterns

### When to use this Tier
- **Large-scale problems** where traditional methods are too slow
- **Repeated similar problems** where learning accumulates
- **Data-rich environments** with historical solution data
- **Strategic planning** requiring high-quality solutions consistently
- **Professional applications** where solution speed and quality matter
- **Research settings** exploring ML-augmented optimization
- **Dynamic environments** where patterns change over time